Importing Libraries


In [29]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from gymnasium import spaces


Preprocessing of House Load and Meta Data

In [30]:
# Load data
load_df = pd.read_csv("House31.csv")

# Convert Date_Time to datetime format
load_df['Date_Time'] = pd.to_datetime(load_df['Date_Time'])

# Set Date_Time as index
load_df.set_index('Date_Time', inplace=True)

# Optional: Resample to 1-minute frequency if raw data is per-second
# load_df = load_df.resample('1min').mean()

In [31]:
# Check for missing or negative values
load_df = load_df.ffill() # Or: load_df.dropna()
load_df = load_df[load_df >= 0].dropna()  # Remove any negative values


In [32]:
# Combine all AC loads
load_df['Total_AC_kW'] = load_df[['AC_LR_kW', 'AC_DR_kW', 'AC_BR_kW']].sum(axis=1)

# Optional: Drop individual AC columns if you don't need them
# load_df.drop(['AC_LR_kW', 'AC_DR_kW', 'AC_BR_kW'], axis=1, inplace=True)


In [33]:
# Only apply on numeric columns
scaler = MinMaxScaler()
scaled_values = scaler.fit_transform(load_df)

load_df_scaled = pd.DataFrame(scaled_values, index=load_df.index, columns=load_df.columns)


In [34]:
meta_df = pd.read_csv("Metadata.csv")

# Clean column names
meta_df.columns = [col.strip().replace(" ", "_").replace("__", "_") for col in meta_df.columns]

# Convert Yes/No to 1/0
meta_df['Ceiling_Insulation'] = meta_df['Ceiling_Insulation'].map({'Yes': 1, 'No': 0})

# Extract values as a dict
meta_features = meta_df.iloc[0].to_dict()
print(meta_features)


{'Website_Name': 'House 1', 'Property_Area_sqft': 4628.26, 'No_of_Floors': 2, 'Building_Year': 2014, 'Ceiling_Height_ft': 9.5, 'Ceiling_Insulation': 1.0, 'Total_No_of_Rooms': 7, 'Bedrooms': 3.0, 'Livingrooms': 2.0, 'Drawingrooms': 1.0, 'Kitchen': 1.0, 'Connection_Type': '3 phase', 'No_of_People': 5, 'Adults_14_to_60': 5, 'Children_0_to_13': 0, 'Seniors': 0, 'Permanent_Residents': 3, 'Temporary_Residents': 2, 'No_of_ACs': 3, 'No_of_Refrigerators': 1, 'No_of_WashingMachines': 1, 'No_of_Electronic_Devices': 6, 'No_of_Fans': 8, 'No_of_Water_Dispensers': 0, 'No_of_Water_Pumps': 2, 'No_of_Electric_Heaters': 3, 'No_of_Irons': 2, 'No_of_Lighting_Devices': 105, 'No_of_UPS': 1}


 Preprocessing: Solar Measurements

In [35]:
# Load the solar dataset
solar_df = pd.read_csv("solar.csv", low_memory=False)

# Convert 'time' to datetime
solar_df['time'] = pd.to_datetime(solar_df['time'])

# Set as index
solar_df.set_index('time', inplace=True)

# Sort by time (optional but safe)
solar_df.sort_index(inplace=True)

In [36]:
# Fill missing values (forward-fill is suitable here for weather data)
solar_df = solar_df.ffill()

# Optionally drop any remaining NA
solar_df.dropna(inplace=True)

# Remove invalid (e.g. negative) solar measurements if needed
solar_cols = ['ghi_pyr', 'ghi_rsi', 'dni', 'dhi']
solar_df[solar_cols] = solar_df[solar_cols].clip(lower=0)


HEMS Environment

In [37]:
# Electricity price structure (PKR per kWh)
hourly_price = {  # Static fixed rates
    0: 20, 1: 20, 2: 20, 3: 20, 4: 20, 5: 20,
    6: 25, 7: 30, 8: 35, 9: 35, 10: 35, 11: 35,
    12: 30, 13: 25, 14: 25, 15: 25, 16: 30, 17: 35,
    18: 35, 19: 35, 20: 30, 21: 25, 22: 22, 23: 20
}

class HEMSEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.load_data = load_df
        self.solar_data = solar_df
        self.price_data = hourly_price
        self.current_step = 0
        self.max_steps = len(self.load_data)

        # System specs
        self.battery_capacity = 5.0  # kWh
        self.battery_soc = 2.5  # Start with 50%
        self.battery_charge_rate = 1.0  # kW
        self.battery_discharge_rate = 1.0  # kW
        self.ups_capacity = 1.0  # kWh, max usage per step
        self.ups_energy = 1.0

        # Action space: [battery_action, ups_action]
        self.action_space = spaces.MultiDiscrete([3, 2])  # battery: charge/discharge/none, UPS: use or not
        # Observation space: [load, solar, SOC, UPS, hour]
        self.observation_space = spaces.Box(
            low=0, high=10, shape=(5,), dtype=np.float32
        )

    def _get_obs(self):
        timestamp = self.load_data.index[self.current_step]
        hour = timestamp.hour
        return np.array([
            self.load_data["Usage_kW"].iloc[self.current_step],
            self.solar_data["Solar_kW"].iloc[self.current_step],
            self.battery_soc,
            self.ups_energy,
            hour / 23.0  # normalize hour
        ], dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.battery_soc = 2.5
        self.ups_energy = 1.0
        return self._get_obs(), {}

    def step(self, action):
        battery_act, ups_act = action
        load = self.load_data["Usage_kW"].iloc[self.current_step]
        solar = self.solar_data["Solar_kW"].iloc[self.current_step]
        hour = self.load_data.index[self.current_step].hour
        price = self.price_data[hour]

        solar_supplied = min(solar, load)
        remaining_load = load - solar_supplied

        battery_supplied = 0.0
        ups_supplied = 0.0
        grid_supplied = 0.0

        if battery_act == 1:  # discharge
            power = min(self.battery_discharge_rate, self.battery_soc, remaining_load)
            self.battery_soc -= power
            battery_supplied = power
            remaining_load -= power
        elif battery_act == 0:  # charge
            surplus_solar = max(0, solar - load)
            charge_power = min(surplus_solar, self.battery_charge_rate, self.battery_capacity - self.battery_soc)
            self.battery_soc += charge_power

        if ups_act == 1 and self.ups_energy > 0 and remaining_load > 0:
            ups_power = min(self.ups_energy, remaining_load)
            self.ups_energy -= ups_power
            ups_supplied = ups_power
            remaining_load -= ups_power

        if remaining_load > 0:
            grid_supplied = remaining_load

        cost = grid_supplied * price
        reward = -cost  # Negative cost

        # Prepare observation
        self.current_step += 1
        terminated = self.current_step >= self.max_steps
        truncated = False
        obs = self._get_obs()

        info = {
            "solar_used": float(solar_supplied),
            "battery_used": float(battery_supplied),
            "ups_used": float(ups_supplied),
            "grid_used": float(grid_supplied),
            "battery_soc": float(self.battery_soc),
            "cost": float(cost)
        }

        return obs, reward, terminated, truncated, info

def plot_combined_env_profiles(env, steps=144):
    obs = env.reset()
    
    solar_gen = []
    load_demand = []
    prices = []
    soc = []

    for _ in range(steps):
        solar_gen.append(env.current_solar_gen)
        load_demand.append(env.current_load)
        prices.append(env.current_price)
        soc.append(env.soc)
        
        _, _, done, _, _ = env.step(env.action_space.sample())
        if done:
            break

    # Create combined plots
    fig, axs = plt.subplots(4, 1, figsize=(12, 12), sharex=True)

    axs[0].plot(solar_gen, label='Solar Generation (kW)')
    axs[0].set_ylabel("Solar (kW)")
    axs[0].set_title("Solar Power Generation")
    axs[0].grid(True)
    axs[0].legend()

    axs[1].plot(load_demand, label='Load Demand (kW)', color='red')
    axs[1].set_ylabel("Load (kW)")
    axs[1].set_title("Electric Load Profile")
    axs[1].grid(True)
    axs[1].legend()

    axs[2].plot(prices, label='Electricity Price (PKR/kWh)', color='purple')
    axs[2].set_ylabel("Price")
    axs[2].set_title("Electricity Pricing")
    axs[2].grid(True)
    axs[2].legend()

    axs[3].plot(soc, label='Battery SoC (%)', color='orange')
    axs[3].set_ylabel("Battery SoC (%)")
    axs[3].set_title("Battery State of Charge")
    axs[3].set_xlabel("Timesteps (10-min intervals)")
    axs[3].grid(True)
    axs[3].legend()

    plt.tight_layout()
    plt.show()


PPO Training

In [39]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import EvalCallback

# Training and Evaluation Envs
env = DummyVecEnv([lambda: HEMSEnv()])
eval_env = DummyVecEnv([lambda: HEMSEnv()])

# Optional: Check environment compliance
check_env(HEMSEnv(), warn=True)

# Create PPO model
model = PPO("MlpPolicy", env, verbose=1, tensorboard_log="./ppo_tensorboard/")

# Setup evaluation callback
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path="./ppo_eval_checkpoints/",
    log_path="./ppo_eval_logs/",
    eval_freq=10_000,
    deterministic=True,
    render=False
)

# Train the model with evaluation callback
model.learn(total_timesteps=100_000, callback=eval_callback)

# Save final model
model.save("ppo_hems_model")




KeyError: 'Solar_kW'

 Visualization


In [ ]:
def plot_ppo_agent_behavior(env, model, title="PPO Agent Behavior After Training"):
    import matplotlib.pyplot as plt

    solar_gen = []
    load_demand = []
    electricity_price = []
    battery_soc = []
    grid_energy = []
    battery_energy = []
    ups_energy = []
    solar_used = []

    obs = env.reset()
    steps = 144  # Visualize 1 day

    for _ in range(steps):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env.step(action)

        solar_gen.append(env.current_solar_gen)
        load_demand.append(env.current_load)
        electricity_price.append(env.current_price)
        battery_soc.append(env.soc)
        grid_energy.append(env.grid_used)
        battery_energy.append(env.battery_used)
        ups_energy.append(env.ups_used)
        solar_used.append(env.solar_used)

    plt.figure(figsize=(14, 12))

    plt.subplot(4, 1, 1)
    plt.plot(solar_gen, label="Solar Generation (kW)")
    plt.plot(solar_used, label="Solar Used (kW)")
    plt.legend()

    plt.subplot(4, 1, 2)
    plt.plot(load_demand, label="Load Demand (kW)")
    plt.plot(grid_energy, label="Grid Used (kW)")
    plt.plot(battery_energy, label="Battery Used (kW)")
    plt.plot(ups_energy, label="UPS Used (kW)")
    plt.legend()

    plt.subplot(4, 1, 3)
    plt.plot(battery_soc, label="Battery SoC (kWh)")
    plt.legend()

    plt.subplot(4, 1, 4)
    plt.plot(electricity_price, label="Electricity Price (PKR/kWh)")
    plt.legend()

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()